### Planned changes so far:
- Transformer to GPT
- Coordinate aware embeddings
- see Email in research log.docx
    - i mean the goal is to construct a better gpt model
        - but also a better training in general thats why i will run tests with a direct comparison between the profs and  my model (his training and eval) and then also change metricies like the Learningratescheduler and see if i can achieve higher performance

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import math
import copy

In [ ]:
class LearnedSessionAdapter(nn.Module):
    """Coordinate-aware spike embeddings + per-task heads.

    - Shared MLPs (coord_mlp_in / coord_mlp_out) dynamically generate 
      w_in (N, d) and w_out (N, d) from 3D unit coordinates.
    - Per-session b_out (N,) bias initialized to mean firing rate.
    - Per-task embedding (d,) to condition the model on the current task.
    """
    def __init__(self, n_units_per_session, task_specs, d, device,
                 mean_rates=None, session_coords=None):
        super().__init__()
        self.d = d
        self.session_coords = [
            torch.tensor(c, dtype=torch.float32, device=device)
            for c in session_coords
        ]
        
        from gradient_normalizer import GradientNormalizer
        self.grad_normalizers = nn.ModuleDict({
            spec.name: GradientNormalizer(normalizer_shape=[1], scale=1.0)
            for spec in task_specs
        })
        self.task_embeds = nn.ParameterDict({
            spec.name: nn.Parameter(torch.randn(d, device=device) * 0.02)
            for spec in task_specs
        })
        
        # coordnate aware mlp
        self.coord_mlp_in = nn.Sequential(
            nn.Linear(3, 64),
            nn.GELU(),
            nn.Linear(64, d),
        )
        
        self.coord_mlp_out = nn.Sequential(
            nn.Linear(3, 64),
            nn.GELU(),
            nn.Linear(64, d),
        )

        # b_out initialized so that relu(0 + b_out) = mean firing rate per neuron
        self._b_out = nn.ParameterList([
            nn.Parameter(torch.from_numpy(mean_rates[i]).float().to(device))
            for i in range(len(n_units_per_session))
        ])
        # Per-task prediction heads
        task_heads = {}
        for spec in task_specs:
            if isinstance(spec, RegressionTask):
                task_heads[spec.name] = nn.Parameter(
                    torch.randn(spec.d_out, d, device=device) * (1.0 / d**0.5))
            elif isinstance(spec, ClassificationTask):
                task_heads[spec.name] = nn.Parameter(
                    torch.randn(spec.n_classes, d, device=device) * (1.0 / d**0.5))
        self.task_heads = nn.ParameterDict(task_heads)

    def w_in(self, session, visible_idx=None):
        coords = self.session_coords[session]
        if visible_idx is not None:
            coords = coords[visible_idx]
        return self.coord_mlp_in(coords)

    def w_out(self, session):
        return self.coord_mlp_out(self.session_coords[session])

    def b_out(self, session):
        return self._b_out[session]

In [ ]:
# schau dir alle klassen an, welche in super keinen klassennamen
# mehr drinnen haben: die hab ich bereits geändert die anderen
# nicht. in ai studio wird der code diskutiert


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        # Ensure that the model dimension (d_model) is divisible by the number of heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        # Initialize dimensions
        self.d_model = d_model # Model's dimension
        self.num_heads = num_heads # Number of attention heads
        self.d_k = d_model // num_heads # Dimension of each head's key, query, and value
        
        # Linear layers for transforming inputs
        self.W_q = nn.Linear(d_model, d_model) # Query transformation
        self.W_k = nn.Linear(d_model, d_model) # Key transformation
        self.W_v = nn.Linear(d_model, d_model) # Value transformation
        self.W_o = nn.Linear(d_model, d_model) # Output transformation
        
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        # Calculate attention scores
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # Apply mask if provided (useful for preventing attention to certain parts like padding)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        
        # Softmax is applied to obtain attention probabilities
        attn_probs = torch.softmax(attn_scores, dim=-1)
        
        # Multiply by values to obtain the final output
        output = torch.matmul(attn_probs, V)
        return output
        
    def split_heads(self, x):
        # Reshape the input to have num_heads for multi-head attention
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)
        
    def combine_heads(self, x):
        # Combine the multiple heads back to original shape
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)
        
    def forward(self, Q, K, V, mask=None):
        # Apply linear transformations and split heads
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))
        
        # Perform scaled dot-product attention
        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Combine heads and apply output transformation
        output = self.W_o(self.combine_heads(attn_output))
        return output

In [ ]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.fc1 = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.fc2 = nn.Linear(4 * config.n_embd, config.n_embd)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.fc2(self.dropout(self.relu(self.fc1(x))))

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super(PositionalEncoding, self).__init__()
        
        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe.unsqueeze(0))
        
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

Below is the gpt block that replaces encoder and decoder from traditional transformer

In [ ]:
class GPTBlock(nn.Module):
    """Der Decoder-Block: Verknüpft Attention und MLP."""
    def __init__(self, config):
        super().__init__()
        self.self_attn = CustomMultiHeadAttention(config)
        self.feed_forward = PositionWiseFeedForward(config)
        
        # DataCamp nutzt LayerNorm (dein Prof nutzte RMSNorm)
        self.norm1 = nn.LayerNorm(config.n_embd)
        self.norm2 = nn.LayerNorm(config.n_embd)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x, block_mask, positions=None):
        # Attention (mit Pre-Norm für bessere Stabilität im Training)

        # positions wird nicht verwendet weil Die Positionen werden 
        # ein einziges Mal ganz am Anfang (in der SpikeGPT-Klasse) 
        # auf den Input x addiert
        attn_out = self.self_attn(self.norm1(x), block_mask)
        x = x + self.dropout(attn_out)
        
        # MLP
        ff_out = self.feed_forward(self.norm2(x))
        x = x + self.dropout(ff_out)
        return x

In [ ]:
class SpikeGPT(nn.Module):
    """Die Hauptklasse. Schnittstelle ist 100% identisch mit der des Profs."""
    def __init__(self, config):
        super().__init__()
        self.config = config
        
        # Das SOS Token Embed (Muss da sein, die Trainingsschleife greift darauf zu!)
        self.sos_embed = nn.Parameter(torch.randn(config.n_embd) * 0.02)
        
        self.positional_encoding = CustomPositionalEncoding(config)
        self.blocks = nn.ModuleList([GPTBlock(config) for _ in range(config.n_layer)])
        self.final_norm = nn.LayerNorm(config.n_embd)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x, block_mask, positions=None):
        # Falls keine Positionen übergeben werden, nehmen wir sture Indizes 0, 1, 2...
        if positions is None:
            positions = torch.arange(x.size(1), device=x.device, dtype=torch.float)

        x = self.positional_encoding(x, positions)
        x = self.dropout(x)

        for block in self.blocks:
            x = block(x, block_mask, positions)

        return self.final_norm(x)

In [ ]:
class SpikeGPTConfig:
    def __init__(self, n_layer=8, n_head=6, n_embd=384, window_size=128, dropout=0.1):
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_embd = n_embd
        self.window_size = window_size
        self.dropout = dropout

In [ ]:
CONFIGS = {
    "tiny": SpikeGPTConfig(n_layer=8, n_head=6, n_embd=384, window_size=32),
    "small": SpikeGPTConfig(n_layer=12, n_head=12, n_embd=768, window_size=64),
    "medium": SpikeGPTConfig(n_layer=24, n_head=16, n_embd=1024, window_size=128),
}

# Training the Transformer model
### Sample Data preperation

In [ ]:
src_vocab_size = 5000
tgt_vocab_size = 5000
d_model = 512
num_heads = 8
num_layers = 6
d_ff = 2048
max_seq_length = 100
dropout = 0.1

transformer = Transformer(src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout)

# Generate random sample data
src_data = torch.randint(1, src_vocab_size, (64, max_seq_length))  # (batch_size, seq_length)
tgt_data = torch.randint(1, tgt_vocab_size, (64, max_seq_length))  # (batch_size, seq_length)

### Training the Model

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(transformer.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)

transformer.train()

for epoch in range(100):
    optimizer.zero_grad()
    output = transformer(src_data, tgt_data[:, :-1])
    loss = criterion(output.contiguous().view(-1, tgt_vocab_size), tgt_data[:, 1:].contiguous().view(-1))
    loss.backward()
    optimizer.step()
    print(f"Epoch: {epoch+1}, Loss: {loss.item()}")

### Transformer model performance eval

In [ ]:
transformer.eval()

# Generate random sample validation data
val_src_data = torch.randint(1, src_vocab_size, (64, max_seq_length))  # (batch_size, seq_length)
val_tgt_data = torch.randint(1, tgt_vocab_size, (64, max_seq_length))  # (batch_size, seq_length)

with torch.no_grad():

    val_output = transformer(val_src_data, val_tgt_data[:, :-1])
    val_loss = criterion(val_output.contiguous().view(-1, tgt_vocab_size), val_tgt_data[:, 1:].contiguous().view(-1))
    print(f"Validation Loss: {val_loss.item()}")